In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Load your raw dataset
df = pd.read_csv(r'D:\Uni\Year_3_sem_2\XAI\Project\phase_2\Dataset.csv') 

# For this example, let's assume 'Patient_ID' and 'Hour' (or 'ICULOS') are your time markers
print(f"Original Data Shape: {df.shape}")

Original Data Shape: (1552210, 44)


In the DeepSEPS paper, they deal with "real-world ICU data" where labs are infrequent. Using a simple mean for everything is a mistake in time-series because it "leaks" future information into the past.

Forward Fill (ffill): If a patient had a temperature taken at Hour 1, we assume it stays similar until the next reading at Hour 4.

Back Fill (bfill): If the very first hour is missing, we use the first available record.

In [16]:
# Sort to ensure time is linear per patient
df = df.sort_values(by=['Patient_ID', 'Hour'])

# Apply Forward Fill per patient
df = df.groupby('Patient_ID').apply(lambda x: x.ffill().bfill())

# If any columns are still NaN (because that specific test was NEVER taken for a patient)
# we use the global median so the model doesn't crash.
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

C:\Users\nour1\AppData\Local\Temp\ipykernel_2720\2845537207.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Patient_ID').apply(lambda x: x.ffill().bfill())


Deep Learning models like DeepSeps are highly sensitive to the magnitude of numbers. If Heart Rate is 100 and pH is 7.4, the model will ignore the pH because the Heart Rate "looks" more important mathematically.

In [17]:
features = df_imputed.drop(columns=['SepsisLabel', 'Patient_ID', 'Hour'])
labels = df_imputed[['Patient_ID', 'SepsisLabel']]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

df_scaled = pd.DataFrame(scaled_features, columns=features.columns)
df_scaled['Patient_ID'] = labels['Patient_ID'].values
df_scaled['SepsisLabel'] = labels['SepsisLabel'].values

In [18]:
# List all potential features
# We exclude ID, Target, and non-medical indices (Unnamed: 0, Hour)
all_features = [f for f in df_scaled.columns if f not in ['Patient_ID', 'SepsisLabel', 'Unnamed: 0', 'Hour']]

print(f"Starting with {len(all_features)} potential clinical features.")

Starting with 40 potential clinical features.


In [19]:
from sklearn.ensemble import RandomForestClassifier

# Use a representative sample for the selection process
df_sample = df_scaled.sample(frac=0.2, random_state=42)
X_sample = df_sample[all_features]
y_sample = df_sample['SepsisLabel']

print("Training Random Forest to identify predictive power...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

# Extract Top 25 features
importances = pd.Series(rf.feature_importances_, index=all_features)
selected_by_importance = importances.nlargest(25).index.tolist()

# GUARANTEE: Ensure ICULOS is treated as 'important' even if it falls outside Top 25
if 'ICULOS' not in selected_by_importance:
    selected_by_importance.append('ICULOS')

print(f"Top 5 Importance: {importances.nlargest(5).index.tolist()}")

Training Random Forest to identify predictive power...
Top 5 Importance: ['ICULOS', 'Temp', 'WBC', 'Platelets', 'HR']


In [20]:
#  Redundancy Filtering by Target Correlation

# 1. Calculate correlations between all features
corr_matrix = df_scaled[all_features].corr().abs()

# 2. Calculate correlation of each feature with the Target (SepsisLabel)
target_corr = df_scaled[all_features + ['SepsisLabel']].corr()['SepsisLabel'].abs().drop('SepsisLabel')

# 3. Identify redundant pairs (Corr > 0.85)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = set()

# Iterate through the pairs
for col in upper.columns:
    # Find features correlated with 'col'
    correlated_features = upper.index[upper[col] > 0.85].tolist()
    
    for feat in correlated_features:
        # We have a redundant pair: (col, feat)
        # Decision: Drop the one that has the lower correlation with the SepsisLabel
        if target_corr[col] > target_corr[feat]:
            to_drop_corr.add(feat)
        else:
            to_drop_corr.add(col)

# 4. PROTECT ICULOS: Ensure it is NEVER in the drop set
to_drop_corr.discard('ICULOS') 

# 5. Create list of non-redundant features
features_no_corr = [f for f in all_features if f not in to_drop_corr]

print(f"Identified {len(to_drop_corr)} redundant features for removal.")
print(f"Features kept based on higher target correlation: {len(features_no_corr)}")

Identified 2 redundant features for removal.
Features kept based on higher target correlation: 38


In [21]:
# The final list is the intersection of Importance and Non-Redundancy
# We add a hard-coded check to ensure ICULOS and vital signs remain
final_feature_list = [f for f in selected_by_importance if f in features_no_corr or f == 'ICULOS']

# Final Safety Check: Remove any accidental non-medical columns
final_feature_list = [f for f in final_feature_list if f != 'Unnamed: 0']

print(f"Feature selection complete.")
print(f"Final Count: {len(final_feature_list)}")
print(f"Selected Features: {final_feature_list}")

Feature selection complete.
Final Count: 24
Selected Features: ['ICULOS', 'Temp', 'WBC', 'Platelets', 'HR', 'Age', 'Glucose', 'SBP', 'HospAdmTime', 'MAP', 'Resp', 'Creatinine', 'DBP', 'BUN', 'Hgb', 'Calcium', 'PTT', 'Potassium', 'Phosphate', 'Magnesium', 'PaCO2', 'Chloride', 'pH', 'Lactate']


In [22]:
# Prepare the final dataframe for sequence generation
final_cols = ['Patient_ID', 'SepsisLabel'] + final_feature_list
df_final_selected = df_scaled[final_cols]

print(f"New Dataframe Shape: {df_final_selected.shape}")
# You can now pass df_final_selected into your create_sequences function!

New Dataframe Shape: (1552210, 26)


The DeepSEPS paper focuses on early prediction. We need to create "windows" of time. If a patient is in the ICU for 40 hours, we might only want to look at 24-hour chunks to predict if they will get sepsis in the next 6 hours.

In [23]:
def create_sequences(data, window_size=24):
    sequences = []
    targets = []
    
    for pid, group in data.groupby('Patient_ID'):
        if len(group) >= window_size:
            # Take the last 'window_size' hours of data
            seq = group.drop(columns=['Patient_ID', 'SepsisLabel']).tail(window_size).values
            label = group['SepsisLabel'].iloc[-1] # Target is the latest status
            
            sequences.append(seq)
            targets.append(label)
            
    return np.array(sequences), np.array(targets)

X, y = create_sequences(df_final_selected, window_size=24)
print(f"Data ready for training. Shape: {X.shape}") 
# Expected shape: (Patients, 24 Hours, Number of Features)

Data ready for training. Shape: (30661, 24, 24)


In [24]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# 1. Split into Train and Test first
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Apply SMOTE only to the Training Set
# Flatten for SMOTE
n_train, n_time, n_feat = X_train_raw.shape
X_train_flat = X_train_raw.reshape(n_train, n_time * n_feat)

sm = SMOTE(random_state=42)
X_train_res, y_train = sm.fit_resample(X_train_flat, y_train_raw)

# 3. Reshape Training set back to 3D
X_train = X_train_res.reshape(X_train_res.shape[0], n_time, n_feat)

print(f"SMOTE complete. Train size: {X_train.shape}, Test size: {X_test.shape}")

SMOTE complete. Train size: (46106, 24, 24), Test size: (6133, 24, 24)


In [25]:
import pickle
import os

# Define paths
output_dir = r'D:\Uni\Year_3_sem_2\XAI\Project\phase_2\DeepSeps_final_processed_data'
os.makedirs(output_dir, exist_ok=True)

# Save Train
with open(os.path.join(output_dir, 'deepseps_train_data.pkl'), 'wb') as f:
    pickle.dump({'X_train': X_train, 'y_train': y_train , 'feature_names' : final_feature_list}, f)

# Save Test
with open(os.path.join(output_dir, 'deepseps_test_data.pkl'), 'wb') as f:
    pickle.dump({'X_test': X_test, 'y_test': y_test , 'feature_names' : final_feature_list}, f)

print("Perfected DeepSeps data saved. No leakage detected.")

Perfected DeepSeps data saved. No leakage detected.
